# Assignment 1 -- LLM Evaluation

**Generation model (baseline):** `google/gemma-2-9b-it-fast`

**Companion files:**
- `product_dataset.csv` -- source product data (50 products)
- `assignment_01.xlsx` -- generated descriptions + manual evaluations (Gemma baseline)
- `assignment_01_llama.xlsx` -- Meta-Llama run (Task 4)

---
## Setup

In [ ]:
import os
import time
import re

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY")
if not NEBIUS_API_KEY:
    raise RuntimeError("NEBIUS_API_KEY not found. Add it to .env")

BASE_URL = "https://api.studio.nebius.com/v1/"
client = OpenAI(api_key=NEBIUS_API_KEY, base_url=BASE_URL)

products = pd.read_csv("../product_dataset.csv")
print(f"Loaded {len(products)} products")
products.head(3)

---
# Task 1 -- Define the Evaluation Rubric (15 pts)

A clear, repeatable scoring framework so human and judge model evaluations follow the same rules.

## 1.1 Criterion Definitions

| Criterion | Good (3 pts) | Ok (2 pts) | Bad (1 pt) |
| :--- | :--- | :--- | :--- |
| **Fluency** | Smooth, varied sentence structure; flows naturally; no robotic repetition; no two sentences start with the same word. | 1 instance of awkward phrasing, a repeated sentence opener, or a robotic transition. | 3+ awkward phrases, highly repetitive sentence structures, or bullet-point style output. |
| **Grammar** | 0 spelling, punctuation, or grammatical errors. | 1 minor error (e.g., a missing comma) that does not obscure meaning. | 2+ errors, or any error that makes a sentence difficult to understand. |
| **Tone** | Actively persuasive: uses at least two distinct persuasive techniques (e.g., benefit framing, emotional/aspirational appeal, sensory language, social proof, urgency). | Uses sales-oriented language but relies on only one persuasive technique or describes features without evoking benefit or aspiration (informative but flat). | Dry/robotic/technical-manual style, overly aggressive, or inappropriate. |
| **Length** | Exactly 55--85 words. | 50--54 or 86--90 words. | Outside the range (<50 or >90 words). |
| **Grounding** | Mentions Name, Material, Warranty, AND all structured attributes -- woven naturally into sentences, with zero hallucinations or invented specs. | Mentions Name, Material, and Warranty but omits one or more structured attributes; OR includes 1 minor plausible but unstated detail. | Contradicts any provided field, invents specs, or omits more than one of Name/Material/Warranty. |
| **Cost** | Combined input/output cost < $0.00002 per description. | Cost between $0.00002 and $0.0005 per description. | Cost > $0.0005 per description. |
| **Latency** | Full response generated in < 5 seconds. | Response generated in 5--10 seconds. | Response takes > 10 seconds. |

## 1.2 Pass/Fail Decision Logic

A description receives a **Pass** verdict only if it clears both the go/on rules andc the pass bar.

### A. Go/No-Go Rules (Automatic Failure)
The output is automatically marked **FAIL** if any of the following hold:
- **Grounding is "Bad"** -- any factual hallucination or contradiction of source data.
- **Length is "Bad"** -- output is < 50 words or > 90 words.
- **Grammar is "Bad"** -- 2+ errors, or errors that impede readability.

### B. Cumulative Pass Bar
If all Go/No-Go rules are satisfied, the description must also meet:
- At least **5 "Good" ratings** across the 7 criteria.
- A minimum total score of **17 / 21 points** (good=3, ok=2, bad=1).

---
# Task 2 -- Generate Descriptions (20 pts)

**Model chosen:** `google/gemma-2-9b-it-fast` (Gemma 2 9B Instruct)

**Rationale:** Gemma is the faster of the two allowed models (~1.1 s vs ~5 s for Meta-Llama), keeping latency in the "good" range. Both are ~8–9B models with comparable quality that we'll iterate on in Task 4.

## 2.1 System Prompt

In [ ]:
SYSTEM_PROMPT = """You are a work for an e-commerce retailer. Your job is to write persuasive product descriptions.

Rules:
- Write exactly 50–90 words. Count carefully.
- Be persuasive use  a friendly, credible tone — enthusiastic but not exaggerated.
- Base the description ONLY on the product name, attributes, material, and warranty provided. Do not invent features.
- Write in flowing prose (no bullet points or headers).
- Output only the description text, nothing else."""

print(SYSTEM_PROMPT)

## 2.2 Generation Code

In [ ]:
MODEL = "google/gemma-2-9b-it-fast"
OUTPUT_FILE = "../assignment/assignment_01.xlsx"


def build_user_message(row: pd.Series) -> str:
    return (
        f"Product name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}"
    )


def generate_description(client: OpenAI, user_message: str, model: str) -> dict:
    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.7,
        max_tokens=200,
    )
    latency_ms = (time.time() - start) * 1000

    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": round(latency_ms, 1),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }

In [ ]:
results = []
for i, row in products.iterrows():
    print(f"[{i + 1}/{len(products)}] Generating: {row['product_name']}")
    user_msg = build_user_message(row)
    result = generate_description(client, user_msg, MODEL)
    result["product_name"] = row["product_name"]
    result["Product_attribute_list"] = row["Product_attribute_list"]
    result["material"] = row["material"]
    result["warranty"] = row["warranty"]
    results.append(result)

df = pd.DataFrame(results)
print(f"\nGenerated {len(df)} descriptions")

## 2.3 Save to Excel

Reorder columns (product info first, then generated fields) and add blank rubric columns for Task 3.

In [ ]:
cols = [
    "product_name", "Product_attribute_list", "material", "warranty",
    "generated_description", "latency_ms", "input_tokens", "output_tokens",
    "fluency", "grammar", "tone", "length", "grounding",
    "latency_rating", "cost_rating",
    "final_score",
]
for col in cols:
    if col not in df.columns:
        df[col] = ""

df = df[cols]
df.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
print(f"Saved {len(df)} rows to {OUTPUT_FILE}")
df.head(3)

---
# Task 3 -- Manual (Human) Evaluation (10 pts)

## 3.1 Cost Calculation

**Gemma-2-9b-it-fast pricing (Nebius AI Studio):**

| | Price per 1M tokens |
|---|---|
| Input tokens | $0.03 |
| Output tokens | $0.09 |

Cost per call = `(input_tokens * 0.03 + output_tokens * 0.09) / 1_000_000`

In [ ]:
# Load the evaluated spreadsheet (with manual scores already filled in)
eval_df = pd.read_excel("../submissions/task_03_gemma.xlsx")
print(f"Loaded {len(eval_df)} rows")
eval_df.head(3)

In [ ]:
# Cost calculation
INPUT_PRICE_PER_TOKEN = 0.03 / 1_000_000   # $0.03 per 1M input tokens
OUTPUT_PRICE_PER_TOKEN = 0.09 / 1_000_000   # $0.09 per 1M output tokens

eval_df["cost_usd"] = (
    eval_df["input_tokens"] * INPUT_PRICE_PER_TOKEN
    + eval_df["output_tokens"] * OUTPUT_PRICE_PER_TOKEN
)

print(f"Average cost per description: ${eval_df['cost_usd'].mean():.8f}")
print(f"Total cost for {len(eval_df)} descriptions: ${eval_df['cost_usd'].sum():.6f}")
eval_df[["product_name", "input_tokens", "output_tokens", "cost_usd"]].head(5)

## 3.2 Manual Evaluation

Rated the first 11 products on all 7 criteria using the Task 1 rubric.

`final_score` logic:
1. Go/No-Go rules (Grounding/Length/Grammar = bad → automatic fail)
2. Cumulative pass bar (≥ 5 good AND ≥ 17/21 points)

In [ ]:
# Show the evaluated rows
CRITERIA = ["fluency", "grammar", "tone", "length", "grounding", "latency_rating", "cost_rating"]

evaluated = eval_df[eval_df["final_score"].isin(["pass", "fail"])].copy()
print(f"Manually evaluated: {len(evaluated)} products")
evaluated[["product_name", *CRITERIA, "score", "final_score", "Word Count"]]

## 3.3 Baseline Analysis

In [ ]:
# Count good/ok/bad per criterion
summary = {}
for crit in CRITERIA:
    counts = evaluated[crit].value_counts()
    summary[crit] = {
        "good": counts.get("good", 0),
        "ok": counts.get("ok", 0),
        "bad": counts.get("bad", 0),
    }

summary_df = pd.DataFrame(summary).T
summary_df["good_rate"] = (summary_df["good"] / len(evaluated) * 100).round(1)
print("Rating distribution across evaluated products:\n")
summary_df

In [ ]:
# Pass/fail breakdown
pass_count = (evaluated["final_score"] == "pass").sum()
fail_count = (evaluated["final_score"] == "fail").sum()
print(f"Pass: {pass_count}/{len(evaluated)} ({pass_count/len(evaluated)*100:.0f}%)")
print(f"Fail: {fail_count}/{len(evaluated)} ({fail_count/len(evaluated)*100:.0f}%)")

# Show failure reasons
print("\nFailed products:")
failed = evaluated[evaluated["final_score"] == "fail"]
failed[["product_name", "Word Count", *CRITERIA, "score", "final_score"]]

### Baseline Findings

**Which criteria performed best?**
Fluency, Grammar, Tone, and Latency — all 100% "good". Gemma writes clean, persuasive prose at ~1.1 s/call, well within the <5 s threshold.

**Which criteria performed worst?**
- **Grounding** — 5/10 (50%) "good"; 5 "ok". Frequently omits an attribute or adds a minor unstated detail.
- **Length** — 7/10 "good", 2 hard failures: Samsung S24 Ultra (98 words) and MacBook Air M3 (93 words) triggered the go/no-go rule.
- **Cost** — 7/10 "good", 3/10 "ok" (longer outputs pushed cost above the $0.00002 threshold).

**Overall baseline:** 7/10 pass (70%). Two length failures; Sony WH-1000XM5 failed the cumulative bar (4 good, needed 5).

**Improvement strategy:**
1. Tighten length rule — target 55–85 words explicitly.
2. Add a few-shot example to anchor output length and tone.
3. Label attribute fields in the user message to improve grounding.
4. Lower temperature to improve instruction-following.

---
# Task 4 -- Improvement Cycle (15 pts)

Three approaches to improve on the Gemma baseline (7/10 pass, 70%):

| Approach | Change |
|---|---|
| **4.1 Model Switch** | `Meta-Llama-3.1-8B-Instruct` |
| **4.2 Decoding Parameters** | temperature 0.7→0.3, add top_p=0.85 |
| **4.3 Prompt Engineering** | Tighter length rule + metadata fix + few-shot example |

---
## 4.1 Model Switch: Meta-Llama-3.1-8B-Instruct

### 1. What changed

Only the model — all other settings (prompt, temperature=0.7, max_tokens=200) unchanged.

| Setting | Baseline | This run |
|---|---|---|
| Model | `google/gemma-2-9b-it-fast` | `meta-llama/Meta-Llama-3.1-8B-Instruct` |

### 2. Why expected to help

Llama's instruction-tuning is stronger on structured constraints. In the Gemma baseline, 2 of 3 failures were length violations (>90 words); Llama was expected to respect word-count rules more consistently.

Tradeoff: Llama is ~5× slower (~5,000–9,000 ms vs ~1,100 ms), pushing `latency_rating` from "good" to "ok" for most products. Cost stays comparable.

### 3. New evaluation scores — see analysis below

In [ ]:
LLAMA_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
OUTPUT_FILE_LLAMA = "task_04_llama_base.xlsx"

llama_results = []
for i, row in products.iterrows():
    print(f"[{i + 1}/{len(products)}] Generating: {row['product_name']}")
    user_msg = build_user_message(row)
    result = generate_description(client, user_msg, LLAMA_MODEL)
    result["product_name"] = row["product_name"]
    result["Product_attribute_list"] = row["Product_attribute_list"]
    result["material"] = row["material"]
    result["warranty"] = row["warranty"]
    llama_results.append(result)

llama_df_raw = pd.DataFrame(llama_results)
print(f"\nGenerated {len(llama_df_raw)} descriptions")


In [ ]:
cols = [
    "product_name", "Product_attribute_list", "material", "warranty",
    "generated_description", "latency_ms", "input_tokens", "output_tokens",
    "fluency", "grammar", "tone", "length", "grounding",
    "latency_rating", "cost_rating",
    "final_score",
]
for col in cols:
    if col not in llama_df_raw.columns:
        llama_df_raw[col] = ""

llama_df_raw = llama_df_raw[cols]
llama_df_raw.to_excel(OUTPUT_FILE_LLAMA, index=False, engine="openpyxl")
print(f"Saved {len(llama_df_raw)} rows to {OUTPUT_FILE_LLAMA}")
llama_df_raw.head(3)


In [ ]:
# Load Meta-Llama results (manually evaluated)
llama_df = pd.read_excel("../submissions/task_04_llama.xlsx")
print(f"Loaded {len(llama_df)} rows")

CRITERIA = ["fluency", "grammar", "tone", "length", "grounding", "latency_rating", "cost_rating"]

llama_evaluated = llama_df[llama_df["final_score"].isin(["pass", "fail"])].copy()
print(f"Manually evaluated: {len(llama_evaluated)} products")
llama_evaluated[["product_name", *CRITERIA, "score - points", "final_score", "Word Count"]]

In [ ]:
# Criterion distribution for Meta-Llama
llama_summary = {}
for crit in CRITERIA:
    counts = llama_evaluated[crit].value_counts()
    llama_summary[crit] = {
        "good": counts.get("good", 0),
        "ok": counts.get("ok", 0),
        "bad": counts.get("bad", 0),
    }

llama_summary_df = pd.DataFrame(llama_summary).T
llama_summary_df["good_rate"] = (llama_summary_df["good"] / len(llama_evaluated) * 100).round(1)
print("Meta-Llama rating distribution:\n")
print(llama_summary_df)

# Pass / fail
llama_pass = (llama_evaluated["final_score"] == "pass").sum()
llama_fail = (llama_evaluated["final_score"] == "fail").sum()
print(f"\nPass: {llama_pass}/{len(llama_evaluated)} ({llama_pass/len(llama_evaluated)*100:.0f}%)")
print(f"Fail: {llama_fail}/{len(llama_evaluated)} ({llama_fail/len(llama_evaluated)*100:.0f}%)")

print(f"\nMean score (all evaluated): {llama_evaluated['score'].mean():.1f} / 21")
print(f"\nFailed products:")
llama_evaluated[llama_evaluated["final_score"] == "fail"][
    ["product_name", "Word Count", *CRITERIA, "score"]
]

### 4.1 Results Summary

| Metric | Gemma baseline | Meta-Llama | Change |
|---|---|---|---|
| **Pass rate** | 7/10 (70%) | 9/10 (90%) | **+20%** |
| **Mean score** | ~19.3 / 21 | ~19.5 / 21 | minor |
| fluency good% | 100% | 90% | -10% |
| grammar good% | 100% | 90% | -10% |
| tone good% | 100% | 100% | — |
| length good% | 70% | 70% | — |
| **grounding good%** | 50% | 70% | **+20%** |
| **latency good%** | 100% | 40% | **-60%** |
| cost good% | 70% | 100% | +30% |

**Key findings:**
- **Pass rate +20%:** Llama fixed both length failures (Samsung S24 Ultra: 98→84 words; MacBook Air M3: 93→80 words).
- **Grounding +20%:** Llama weaves structured attributes more naturally.
- **Latency -60%:** Every call dropped to "ok" (5–10 s). Acceptable tradeoff for the pass-rate gain, but would matter in production.
- **Remaining failure:** Garmin Forerunner 255 still fails (92 words) in both models — primary target for Approaches B and C.

In [ ]:
# Approach B generation: Gemma with tighter decoding params
# MODEL = "google/gemma-2-9b-it-fast"   # same model as baseline
# temperature=0.3, top_p=0.85           # changed from temperature=0.7, no top_p

def generate_description_v2(client: OpenAI, user_message: str, model: str) -> dict:
    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.3,   # was 0.7; reduces stochastic drift on length/grounding
        top_p=0.85,        # nucleus sampling; discards low-prob tail tokens
        max_tokens=200,
    )
    latency_ms = (time.time() - start) * 1000
    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": round(latency_ms, 1),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }

print("Approach B generation function defined (not run — results pending).")

---
## 4.2 Decoding Parameters

### 1. What changed

Two parameter changes applied to Gemma (prompt unchanged):

| Parameter | Before | After |
|---|---|---|
| `temperature` | `0.7` | `0.3` |
| `top_p` | *(default 1.0)* | `0.85` |

### 2. Why expected to help

**Temperature 0.7→0.3:** Makes sampling near-deterministic, reducing stochastic drift on length and grounding.

**top_p=0.85:** Nucleus sampling hard-cuts the distribution to the top 85% probability mass, eliminating rare tokens that cause hallucinated specs or runaway sentence length. Together these are belt-and-suspenders: temperature sharpens the distribution, top_p removes the remaining tail.

**Expected gains:** Both length and grounding failures stem from the model generating unexpected tokens — constraining the sampling space should push some "bad" length and "ok" grounding toward "good".

### 3. New evaluation scores

### Code for temprature and top_p

In [ ]:
# Approach B generation: Gemma with tighter decoding params
# MODEL = "google/gemma-2-9b-it-fast"   # same model as baseline
# temperature=0.3, top_p=0.85           # changed from temperature=0.7, no top_p

def generate_description_v2(client: OpenAI, user_message: str, model: str) -> dict:
    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.3,   # was 0.7; reduces stochastic drift on length/grounding
        top_p=0.85,        # nucleus sampling; discards low-prob tail tokens
        max_tokens=200,
    )
    latency_ms = (time.time() - start) * 1000
    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": round(latency_ms, 1),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }


### 4.2 Results Summary — Best Mean Score

| Metric | Gemma baseline | Temp 0.3 + top_p 0.85 | Change |
|---|---|---|---|
| **Pass rate** | 7/10 (70%) | 9/10 (90%) | **+20%** |
| **Mean score** | ~19.3 / 21 | 20.1 / 21 | **+0.8** |
| fluency good% | 100% | 100% | — |
| grammar good% | 100% | 100% | — |
| tone good% | 100% | 100% | — |
| **length good%** | 70% | 80% | **+10%** |
| grounding good% | 50% | 50% | — |
| latency good% | 100% | 100% | — |
| **cost good%** | 70% | 90% | **+20%** |

**Key findings:**
- **Pass rate +20%:** Fixed Samsung S24 Ultra (98→82 words) and MacBook Air M3 (93→78 words) — both previously over 90 words, now cleanly within range.
- **Length improved:** 8/10 "good" vs 7/10. Garmin Forerunner 255 dropped from bad (92 words) to ok (88 words) — still narrowly outside the good band.
- **Cost improved:** 9/10 "good" vs 7/10. Lower temperature reduces token variance, keeping output lengths more consistent and costs below the threshold.
- **Grounding unchanged:** 5/10 "good" — constraining sampling did not help the model better cover structured attributes.
- **Remaining failure:** Sony WH-1000XM5 still fails with length=bad (106 words), triggering the go/no-go rule.

**Takeaway:** Combining temperature=0.3 with top_p=0.85 was more effective than temperature alone. The parameter changes produced the highest mean score across all three approaches (20.1/21), with no regression in fluency, grammar, or tone — making this the strongest overall result.

---
## 4.3 Prompt Engineering

### 1. What changed

Three changes to the system prompt (model and decoding params unchanged):

**a. Tighter length instruction** — added active self-check:
```
Before:  "Write exactly 50–90 words. Count carefully."
After:   "Your description MUST be 50–90 words. Count every word before you finish.
          If your draft exceeds 90 words, shorten it. If under 50, expand it."
```

**b. Metadata attribute fix** — clarified which attribute labels are metadata, not features:
```
Added:   "Some attributes use labels like battery:, weight:, capacity:, etc. — these are
          metadata descriptors. Only mention features that make sense for the product
          category (e.g., a water bottle does not have a battery)."
```

**c. Few-shot example** — one 65-word grounded description appended to anchor length and tone.

### 2. Why expected to help

- **Length:** Two of three Gemma failures were >90 word outputs. An active remediation step ("shorten it before outputting") is more reliable than a passive word-count rule for small models.
- **Grounding:** Both models misinterpret metadata labels on mismatched products (e.g., `battery: long-lasting` on a water bottle). One explicit rule eliminates this class of error.
- **Few-shot:** For 8–9B models, a single in-context example calibrates output length and tone more reliably than instructions alone.


### full prompt:

In [ ]:
SYSTEM_PROMPT_V2 = """You are a work for an e-commerce retailer. Your job is to write persuasive product descriptions.

Rules:
- Your description MUST be 50–90 words. Count every word before you finish.
  If your draft exceeds 90 words, shorten it before outputting.
  If it is under 50 words, expand it.
- Be persuasive — use a friendly, credible tone, enthusiastic but not exaggerated.
- Base the description ONLY on the product name, attributes, material, and warranty provided. Do not invent features.
- Some attributes use labels like battery:, weight:, capacity:, rating:, dimensions:, or color options: — these are
  metadata descriptors, not product features. Only mention features that make sense for the product category
  (e.g., a compact dimension does not need to be mentioned as a feature).
- Write in flowing prose (no bullet points or headers).
- Output only the description text, nothing else.

Example input:
  Product name: OnePlus 12
  Attributes: features: Snapdragon 8 Gen 3, 50 MP Hasselblad camera, 100 W SUPERVOOC charging; dimensions: sleek
  Material: silky glass back, aluminum frame
  Warranty: 1-year limited warranty

Example output:
  Meet the OnePlus 12 — a flagship that refuses to compromise. Powered by the Snapdragon 8 Gen 3,
  it blazes through every task while the 50 MP Hasselblad camera captures stunning detail in any light.
  The 100 W SUPERVOOC charging tops up the battery in minutes, not hours. Wrapped in silky glass with
  an aluminum frame, it's built to impress. Backed by a 1-year limited warranty."""

print(SYSTEM_PROMPT_V2)

In [ ]:
OUTPUT_FILE_PROMPT_V2 = "../assignment/task_04_prompt_v2.xlsx"

# Swap SYSTEM_PROMPT to V2 so generate_description (from Task 2) uses the updated prompt
_orig_prompt = SYSTEM_PROMPT
SYSTEM_PROMPT = SYSTEM_PROMPT_V2

v2_results = []
for i, row in products.iterrows():
    print(f"[{i + 1}/{len(products)}] Generating: {row['product_name']}")
    user_msg = build_user_message(row)
    result = generate_description(client, user_msg, MODEL)
    result["product_name"] = row["product_name"]
    result["Product_attribute_list"] = row["Product_attribute_list"]
    result["material"] = row["material"]
    result["warranty"] = row["warranty"]
    v2_results.append(result)

SYSTEM_PROMPT = _orig_prompt  # restore

v2_df = pd.DataFrame(v2_results)
print(f"\nGenerated {len(v2_df)} descriptions")

# Save with blank rubric columns
cols = [
    "product_name", "Product_attribute_list", "material", "warranty",
    "generated_description", "latency_ms", "input_tokens", "output_tokens",
    "fluency", "grammar", "tone", "length", "grounding",
    "latency_rating", "cost_rating", "final_score",
]
for col in cols:
    if col not in v2_df.columns:
        v2_df[col] = ""
v2_df = v2_df[cols]
v2_df.to_excel(OUTPUT_FILE_PROMPT_V2, index=False, engine="openpyxl")
print(f"Saved to {OUTPUT_FILE_PROMPT_V2}")
v2_df.head(3)

| Metric | Gemma baseline | Prompt V2 | Change |
|---|---|---|---|
| **Pass rate** | 7/10 (70%) | 9/10 (90%) | **+20%** |
| **Mean score** | ~19.3 / 21 | 19.0 / 21 | -0.3 |
| fluency good% | 100% | 100% | — |
| grammar good% | 100% | 70% | **-30%** |
| tone good% | 100% | 100% | — |
| **length good%** | 70% | 100% | **+30%** |
| **grounding good%** | 50% | 30% | **-20%** |
| latency good% | 100% | 100% | — |
| **cost good%** | 70% | 0% | **-70%** |

**Key findings:**
- **Length +30% — biggest win:** 10/10 "good" (up from 7/10). The active self-check instruction fully eliminated length failures!! 
- **Cost -70% — unexpected loss:** 0/10 "good", all "ok". The longer system prompt (with the few-shot example) increases input token count, This is not really an issue since the $0.00002 was an arbirtary choice and current cost ~0.00003$ is still reasnoable.
- **Grammar -30%:** 3 products dropped from "good" to "ok". The expanded prompt may be adding complexity that slightly degrades output grammar consistency.
- **Grounding -20%:** Down from 5/10 to 3/10 "good". we didn't expect that, the few-shot example and additional rules may be crowding out the model's attention to attribute coverage.
- **Remaining failure:** Sony WH-1000XM5 fails the cumulative pass bar — only 4 "good" ratings (grammar=ok, grounding=ok, cost=ok), one short of the required 5.

**Conclusion and summary:**
- **Length and pass percentage** the new prompt really improved the length issue!!
    pass criteria also imporved. 
- **Cost & grounding:** cost is not really signfinant since we chose an artirary   threshold, that can be changed. Grounding and grammar has more meaning. so probably improving the prompt a bit, maybe removing the example input and output might help.

---
## 4.4 Task 4 — Overall Conclusions

### Best Approach
All three approaches matched on pass rate (9/10, 90%), but **decoding parameters (4.2)** produced the highest mean score (20.1/21) with no regression in any quality criterion. It is the strongest result overall.

Prompt engineering (4.3) achieved perfect length scores but degraded grounding and grammar — a tradeoff that is harder to justify. Combining the improved prompt with tighter decoding parameters would likely produce the best of both: better length compliance and better output quality.

### Rubric Calibration
The improvement cycle revealed that the rubric's thresholds matter as much as the model's output. Specifically:

- **Cost and latency** thresholds were set without a real business anchor — they triggered "ok" ratings on otherwise strong outputs. These criteria should be recalibrated to reflect actual production constraints rather than arbitrary cutoffs.
- **Grounding** is the hardest criterion to improve and the most meaningful for a real e-commerce use case. It remained at 50% "good" across all approaches. This is the primary target for future work.

### Next Steps
- Combine prompt V2 with temperature=0.3 / top_p=0.85 and re-evaluate.
- Focus prompt improvements on attribute coverage rather than length, since length is now solved.
- Consider relaxing the cost threshold to reflect realistic pricing.

# Task 5: LLM-as-a-Judge for Product Description Evaluation

We start by implementing an automated judge using the meta-llama/Meta-Llama-3.1-8B-Instructa which was not use in Task 2. 

## 5.1 Setup

In [136]:
import os
import time
import json
from typing import Literal

import pandas as pd
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

# ── Load environment variables ──────────────────────────────────────────
load_dotenv("../.env")

# ── Create OpenAI-compatible client for Nebius AI Studio ────────────────
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

# ── Load human-evaluated baseline ─────────────────────────────────
df = pd.read_excel("assignment_01.xlsx", engine="openpyxl")
print(f"Loaded {len(df)} products from assignment_01.xlsx")
print(f"Columns: {df.columns.tolist()}")
print(f"Products with human eval: {df['fluency'].notna().sum()}")

Loaded 50 products from assignment_01.xlsx
Columns: ['product_name', 'Product_attribute_list', 'material', 'warranty', 'generated_description', 'latency_ms', 'input_tokens', 'output_tokens', 'fluency', 'grammar', 'tone', 'length', 'grounding', 'latency_rating', 'cost_rating', 'final_score']
Products with human eval: 0


## 5.2 Configuration

Change `JUDGE_MODEL` below to swap the judge model without touching any other code.

In [137]:
# ── Judge model (the model NOT used for generation in Task 2) ───────────
# Task 2 used google/gemma-2-9b-it-fast, so the default judge is Llama.
# Switch to e.g. "Qwen/Qwen3-30B-A3B-Instruct-2507" if needed.
JUDGE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"Judge model: {JUDGE_MODEL}")

Judge model: meta-llama/Meta-Llama-3.1-8B-Instruct


## 5.3 Why "Explanation Before Verdict" Matters

In the Pydantic schema below, every criterion result has the `explanation`
field **before** the `verdict` field. This is not arbitrary -- it is a
deliberate design choice that mirrors chain-of-thought prompting.

When a language model generates structured JSON, it produces tokens
left-to-right. If `verdict` came first, the model would commit to a rating
("good", "ok", or "bad") before articulating its reasoning, which encourages
snap judgments and makes the explanation a post-hoc rationalization.

By placing `explanation` first, the model is forced to lay out the evidence
and reasoning *before* it ever emits the verdict token. This:

1. **Reduces hasty errors** -- the model considers the rubric in detail
   before locking in a score.
2. **Improves consistency** -- the explanation grounds the verdict, so
   similar descriptions tend to receive similar ratings.
3. **Aids debugging** -- we can read the explanation to understand *why*
   the judge chose a particular rating and catch systematic biases.

## 5.4 Pydantic Schemas

In [138]:
# ── Schema for all-criteria-at-once judging ─────────────────────────────

class CriterionResult(BaseModel):
    """A single criterion evaluation. Explanation MUST come before verdict."""
    explanation: str   # reasoning first — forces chain-of-thought
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    """Complete judge output for all five text-quality criteria."""
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult


# ── Schema for single-criterion judging ─────────────────────────────────

class SingleCriterionResult(BaseModel):
    """Judge output when evaluating a single criterion in isolation."""
    explanation: str   # reasoning first
    verdict: Literal["good", "ok", "bad"]


# Quick sanity check: print the JSON schema the API will receive
print("JudgeOutput schema (excerpt):")
print(json.dumps(JudgeOutput.model_json_schema(), indent=2)[:600], "...")

JudgeOutput schema (excerpt):
{
  "$defs": {
    "CriterionResult": {
      "description": "A single criterion evaluation. Explanation MUST come before verdict.",
      "properties": {
        "explanation": {
          "title": "Explanation",
          "type": "string"
        },
        "verdict": {
          "enum": [
            "good",
            "ok",
            "bad"
          ],
          "title": "Verdict",
          "type": "string"
        }
      },
      "required": [
        "explanation",
        "verdict"
      ],
      "title": "CriterionResult",
      "type": "object"
    }
  },
  "description": "Comple ...


## 5.5 Judge Prompts

The system prompt embeds the full rubric so the judge has explicit criteria.
The user message supplies the source product data and generated description.

In [139]:
# ── System prompt for ALL-CRITERIA-AT-ONCE judging ──────────────────────

JUDGE_SYSTEM_PROMPT = """\
You are an expert product-copy evaluator. You will receive:
1. SOURCE DATA — the original product information (name, attributes, material, warranty).
2. GENERATED DESCRIPTION — a marketing description produced by an LLM.

Evaluate the description on EXACTLY five criteria using the rubric below.
For EACH criterion, first write a concise explanation of your reasoning,
then give your verdict as one of: "good", "ok", or "bad".

═══ RUBRIC ═══

FLUENCY
  good (3 pts): Smooth, varied sentence structure; flows naturally; no robotic
    repetition; no two sentences start with the same word.
  ok   (2 pts): 1 instance of awkward phrasing, a repeated sentence opener,
    or a robotic transition.
  bad  (1 pt):  3+ awkward phrases, highly repetitive sentence structures,
    or bullet-point style output.

GRAMMAR
  good (3 pts): 0 spelling, punctuation, or grammatical errors.
  ok   (2 pts): 1 minor error (e.g., a missing comma) that does not obscure meaning.
  bad  (1 pt):  2+ errors, or any error that makes a sentence difficult to understand.

TONE
  good (3 pts): Actively persuasive — uses at least two distinct persuasive
    techniques (e.g., benefit framing, emotional/aspirational appeal, sensory
    language, social proof, urgency).
  ok   (2 pts): Uses sales-oriented language but relies on only one persuasive
    technique or describes features without evoking benefit or aspiration
    (informative but flat).
  bad  (1 pt):  Dry/robotic/technical-manual style, overly aggressive, or
    inappropriate.

LENGTH
  good (3 pts): Exactly 55–85 words.
  ok   (2 pts): 50–54 or 86–90 words.
  bad  (1 pt):  Outside the range (<50 or >90 words).

GROUNDING
  good (3 pts): Mentions Name, Material, Warranty, AND all structured
    attributes — woven naturally into sentences, with zero hallucinations or
    invented specs.
  ok   (2 pts): Mentions Name, Material, and Warranty but omits one or more
    structured attributes; OR includes 1 minor plausible but unstated detail.
  bad  (1 pt):  Contradicts any provided field, invents specs, or omits more
    than one of Name/Material/Warranty.

═══ IMPORTANT ═══
- For GROUNDING, compare the description carefully against the SOURCE DATA.
  Flag any detail in the description that is NOT in the source data.
- For LENGTH, you MUST count every word carefully. In your explanation,
  list the words numbered 1 through N (e.g., "1:Discover 2:the 3:future ...").
  After listing, state the total and apply the rubric thresholds.
  Take your time — accuracy matters more than speed.
- Provide your explanation BEFORE your verdict for each criterion.
- Respond ONLY with the requested JSON structure — no extra text.
"""


def build_user_message(row: pd.Series) -> str:
    """Build the user message containing source data + generated description."""
    return f"""\
═══ SOURCE DATA ═══
Product Name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}

═══ GENERATED DESCRIPTION ═══
{row['generated_description']}
"""


print("System prompt length:", len(JUDGE_SYSTEM_PROMPT), "chars")
print("Example user message:")
print(build_user_message(df.iloc[0])[:300], "...")

System prompt length: 2674 chars
Example user message:
═══ SOURCE DATA ═══
Product Name: Apple iPhone 15 Pro
Attributes: features: A17 Pro chip, 120 Hz ProMotion display, USB‑C fast charging; dimensions: compact
Material: titanium frame, Ceramic Shield glass
Warranty: 1‑year limited warranty

═══ GENERATED DESCRIPTION ═══
Experience the power and precis ...


In [140]:
def judge_all_criteria(
    row: pd.Series,
    client: OpenAI,
    model: str = JUDGE_MODEL,
) -> dict:
    """Call the judge model once to evaluate all 5 criteria for a single product."""
    user_msg = build_user_message(row)

    t0 = time.time()
    response = client.chat.completions.create(
        model=model,
        temperature=0.3,
        top_p=0.95,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "JudgeOutput",
                "schema": JudgeOutput.model_json_schema()
            },
        },
    )
    latency_ms = (time.time() - t0) * 1000

    raw = response.choices[0].message.content
    parsed = json.loads(raw)

    result = {}
    for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
        result[f"{criterion}_explanation"] = parsed[criterion]["explanation"]
        result[f"{criterion}_verdict"] = parsed[criterion]["verdict"]

    result["judge_latency_ms"] = round(latency_ms, 1)
    result["judge_input_tokens"] = response.usage.prompt_tokens
    result["judge_output_tokens"] = response.usage.completion_tokens

    return result


print("judge_all_criteria() defined.")

judge_all_criteria() defined.


In [141]:
# ── Rubric text for single-criterion prompts ───────────────────────────

CRITERION_RUBRICS = {
    "fluency": """\
FLUENCY
  good (3 pts): Smooth, varied sentence structure; flows naturally; no robotic
    repetition; no two sentences start with the same word.
  ok   (2 pts): 1 instance of awkward phrasing, a repeated sentence opener,
    or a robotic transition.
  bad  (1 pt):  3+ awkward phrases, highly repetitive sentence structures,
    or bullet-point style output.""",

    "grammar": """\
GRAMMAR
  good (3 pts): 0 spelling, punctuation, or grammatical errors.
  ok   (2 pts): 1 minor error (e.g., a missing comma) that does not obscure meaning.
  bad  (1 pt):  2+ errors, or any error that makes a sentence difficult to understand.""",

    "tone": """\
TONE
  good (3 pts): Actively persuasive — uses at least two distinct persuasive
    techniques (e.g., benefit framing, emotional/aspirational appeal, sensory
    language, social proof, urgency).
  ok   (2 pts): Uses sales-oriented language but relies on only one persuasive
    technique or describes features without evoking benefit or aspiration
    (informative but flat).
  bad  (1 pt):  Dry/robotic/technical-manual style, overly aggressive, or
    inappropriate.""",

    "length": """\
LENGTH
  good (3 pts): Exactly 55–85 words.
  ok   (2 pts): 50–54 or 86–90 words.
  bad  (1 pt):  Outside the range (<50 or >90 words).""",

    "grounding": """\
GROUNDING
  good (3 pts): Mentions Name, Material, Warranty, AND all structured
    attributes — woven naturally into sentences, with zero hallucinations or
    invented specs.
  ok   (2 pts): Mentions Name, Material, and Warranty but omits one or more
    structured attributes; OR includes 1 minor plausible but unstated detail.
  bad  (1 pt):  Contradicts any provided field, invents specs, or omits more
    than one of Name/Material/Warranty.""",
}


def _single_criterion_system_prompt(criterion_name: str, rubric_text: str) -> str:
    """Build a system prompt focused on a single criterion."""
    grounding_note = ""
    if criterion_name == "grounding":
        grounding_note = (
            "\nIMPORTANT: Compare the description carefully against the "
            "SOURCE DATA. Flag any detail that is NOT in the source data.\n"
        )
    length_note = ""
    if criterion_name == "length":
        length_note = (
            "\nIMPORTANT: You MUST count every word carefully. In your explanation, "
            "list the words numbered 1 through N (e.g., \"1:Discover 2:the 3:future ...\"). "
            "After listing, state the total and apply the rubric thresholds. "
            "Take your time — accuracy matters more than speed.\n"
        )
    return f"""\
You are an expert product-copy evaluator. You will receive:
1. SOURCE DATA — the original product information (name, attributes, material, warranty).
2. GENERATED DESCRIPTION — a marketing description produced by an LLM.

Evaluate the description on the following criterion ONLY:

{rubric_text}
{grounding_note}{length_note}
First, write a concise explanation of your reasoning.
Then, give your verdict as one of: "good", "ok", or "bad".
Respond ONLY with the requested JSON structure — no extra text.
"""


def judge_single_criterion(
    row: pd.Series,
    criterion_name: str,
    client: OpenAI,
    model: str = JUDGE_MODEL,
) -> dict:
    """Call the judge model to evaluate ONE criterion for a single product."""
    rubric_text = CRITERION_RUBRICS[criterion_name]
    system_prompt = _single_criterion_system_prompt(criterion_name, rubric_text)
    user_msg = build_user_message(row)

    t0 = time.time()
    response = client.chat.completions.create(
        model=model,
        temperature=0.3,
        top_p=0.95,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SingleCriterionResult",
                "schema": SingleCriterionResult.model_json_schema()
            },
        },
    )
    latency_ms = (time.time() - t0) * 1000

    raw = response.choices[0].message.content
    parsed = json.loads(raw)

    return {
        "explanation": parsed["explanation"],
        "verdict": parsed["verdict"],
        "judge_latency_ms": round(latency_ms, 1),
        "judge_input_tokens": response.usage.prompt_tokens,
        "judge_output_tokens": response.usage.completion_tokens,
    }


print("judge_single_criterion() defined.")
print("Criteria available:", list(CRITERION_RUBRICS.keys()))

judge_single_criterion() defined.
Criteria available: ['fluency', 'grammar', 'tone', 'length', 'grounding']


## 5.7 Pass/Fail Logic

The same rules from Task 1 apply here. Cost and Latency ratings come from
the existing columns in the data (they are measured programmatically, not
by the judge).

In [142]:
VERDICT_TO_SCORE = {"good": 3, "ok": 2, "bad": 1}
JUDGE_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]
ALL_CRITERIA = JUDGE_CRITERIA + ["latency_rating", "cost_rating"]


def compute_pass_fail(row: pd.Series, verdict_suffix: str = "_verdict") -> dict:
    """
    Apply pass/fail logic using judge verdicts + programmatic cost/latency.

    Parameters
    ----------
    row : pd.Series
        Must contain columns like fluency_verdict, ..., grounding_verdict,
        plus latency_rating and cost_rating from the original data.
    verdict_suffix : str
        Suffix for judge verdict columns (default '_verdict').

    Returns
    -------
    dict with total_score, num_good, final_verdict
    """
    # Collect all 7 ratings (5 from judge + 2 programmatic)
    ratings = {}
    for c in JUDGE_CRITERIA:
        ratings[c] = row.get(f"{c}{verdict_suffix}", "bad")
    ratings["latency_rating"] = row.get("latency_rating", "bad")
    ratings["cost_rating"] = row.get("cost_rating", "bad")

    # Compute total score
    total_score = sum(VERDICT_TO_SCORE.get(v, 1) for v in ratings.values())
    num_good = sum(1 for v in ratings.values() if v == "good")

    # Go/No-Go auto-fail checks
    auto_fail = (
        ratings.get("grounding") == "bad"
        or ratings.get("length") == "bad"
        or ratings.get("grammar") == "bad"
    )

    if auto_fail:
        final = "fail"
    elif num_good >= 5 and total_score >= 17:
        final = "pass"
    else:
        final = "fail"

    return {
        "total_score": total_score,
        "num_good": num_good,
        "final_verdict": final,
    }


print("compute_pass_fail() defined.")

compute_pass_fail() defined.


---

## 6.0 Restore Session

Run **only this cell** after a kernel restart to restore all state needed for
Task 6 — imports, client, constants, data, and function definitions.
Skip it if the kernel is already warm.

In [143]:
# ════════════════════════════════════════════════════════════════════════
# 6.0  RESTORE SESSION
# Run this single cell after a kernel restart to recreate all state needed
# for Task 6.  Skip if the kernel is already warm.
# ════════════════════════════════════════════════════════════════════════

import os, time, json
from typing import Literal
import pandas as pd
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

# ── Client ───────────────────────────────────────────────────────────────
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

# ── Model constants ───────────────────────────────────────────────────────
JUDGE_MODEL    = "meta-llama/Meta-Llama-3.1-8B-Instruct"
JUDGE_MODEL_V2 = "Qwen/Qwen3-30B-A3B-Instruct-2507"

# ── Data ──────────────────────────────────────────────────────────────────
df       = pd.read_excel("assignment_01.xlsx", engine="openpyxl")
df_human = pd.read_excel("task_03_gemma.xlsx")

# ── Pydantic schemas ──────────────────────────────────────────────────────
class CriterionResult(BaseModel):
    """A single criterion evaluation. Explanation MUST come before verdict."""
    explanation: str
    verdict: Literal["good", "ok", "bad"]

class JudgeOutput(BaseModel):
    """Complete judge output for all five text-quality criteria."""
    fluency:    CriterionResult
    grammar:    CriterionResult
    tone:       CriterionResult
    length:     CriterionResult
    grounding:  CriterionResult

class SingleCriterionResult(BaseModel):
    """Single-criterion judge output. Explanation MUST come before verdict."""
    explanation: str
    verdict: Literal["good", "ok", "bad"]

# ── System prompt & user message builder ─────────────────────────────────
JUDGE_SYSTEM_PROMPT = """\
You are an expert product-copy evaluator. You will receive:
1. SOURCE DATA — the original product information (name, attributes, material, warranty).
2. GENERATED DESCRIPTION — a marketing description produced by an LLM.

Evaluate the description on EXACTLY five criteria using the rubric below.
For EACH criterion, first write a concise explanation of your reasoning,
then give your verdict as one of: "good", "ok", or "bad".

═══ RUBRIC ═══

FLUENCY
  good (3 pts): Smooth, varied sentence structure; flows naturally; no robotic
    repetition; no two sentences start with the same word.
  ok   (2 pts): 1 instance of awkward phrasing, a repeated sentence opener,
    or a robotic transition.
  bad  (1 pt):  3+ awkward phrases, highly repetitive sentence structures,
    or bullet-point style output.

GRAMMAR
  good (3 pts): 0 spelling, punctuation, or grammatical errors.
  ok   (2 pts): 1 minor error (e.g., a missing comma) that does not obscure meaning.
  bad  (1 pt):  2+ errors, or any error that makes a sentence difficult to understand.

TONE
  good (3 pts): Actively persuasive — uses at least two distinct persuasive
    techniques (e.g., benefit framing, emotional/aspirational appeal, sensory
    language, social proof, urgency).
  ok   (2 pts): Uses sales-oriented language but relies on only one persuasive
    technique or describes features without evoking benefit or aspiration
    (informative but flat).
  bad  (1 pt):  Dry/robotic/technical-manual style, overly aggressive, or
    inappropriate.

LENGTH
  good (3 pts): Exactly 55–85 words.
  ok   (2 pts): 50–54 or 86–90 words.
  bad  (1 pt):  Outside the range (<50 or >90 words).

GROUNDING
  good (3 pts): Mentions Name, Material, Warranty, AND all structured
    attributes — woven naturally into sentences, with zero hallucinations or
    invented specs.
  ok   (2 pts): Mentions Name, Material, and Warranty but omits one or more
    structured attributes; OR includes 1 minor plausible but unstated detail.
  bad  (1 pt):  Contradicts any provided field, invents specs, or omits more
    than one of Name/Material/Warranty.

═══ IMPORTANT ═══
- For GROUNDING, compare the description carefully against the SOURCE DATA.
  Flag any detail in the description that is NOT in the source data.
- For LENGTH, you MUST count every word carefully. In your explanation,
list the words numbered 1 through N (e.g., "1:Discover 2:the 3:future ...").
After listing, state the total and apply the rubric thresholds.
Take your time — accuracy matters more than speed.
- Provide your explanation BEFORE your verdict for each criterion.
- Respond ONLY with the requested JSON structure — no extra text.
"""

def build_user_message(row: pd.Series) -> str:
    return f"""\
═══ SOURCE DATA ═══
Product Name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}

═══ GENERATED DESCRIPTION ═══
{row['generated_description']}
"""

# ── Judge functions ───────────────────────────────────────────────────────
def judge_all_criteria(row: pd.Series, client: OpenAI, model: str = JUDGE_MODEL) -> dict:
    user_msg = build_user_message(row)
    t0 = time.time()
    response = client.chat.completions.create(
        model=model, temperature=0.3,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "JudgeOutput", "schema": JudgeOutput.model_json_schema()},
        },
    )
    latency_ms = (time.time() - t0) * 1000
    parsed = json.loads(response.choices[0].message.content)
    result = {}
    for c in ["fluency", "grammar", "tone", "length", "grounding"]:
        result[f"{c}_explanation"] = parsed[c]["explanation"]
        result[f"{c}_verdict"]     = parsed[c]["verdict"]
    result["judge_latency_ms"]    = round(latency_ms, 1)
    result["judge_input_tokens"]  = response.usage.prompt_tokens
    result["judge_output_tokens"] = response.usage.completion_tokens
    return result

CRITERION_RUBRICS = {
    "fluency":   "FLUENCY\n  good (3 pts): Smooth, varied sentence structure; flows naturally.\n  ok   (2 pts): 1 awkward phrase or repeated opener.\n  bad  (1 pt):  3+ awkward phrases or bullet-point style.",
    "grammar":   "GRAMMAR\n  good (3 pts): 0 errors.\n  ok   (2 pts): 1 minor error.\n  bad  (1 pt):  2+ errors.",
    "tone":      "TONE\n  good (3 pts): Two+ distinct persuasive techniques.\n  ok   (2 pts): One persuasive technique.\n  bad  (1 pt):  Dry/robotic/inappropriate.",
    "length":    "LENGTH\n  good (3 pts): Exactly 55–85 words.\n  ok   (2 pts): 50–54 or 86–90 words.\n  bad  (1 pt):  Outside the range (<50 or >90 words).",
    "grounding": "GROUNDING\n  good (3 pts): Name+Material+Warranty+all attributes, zero hallucinations.\n  ok   (2 pts): Omits one attribute OR 1 minor unstated detail.\n  bad  (1 pt):  Contradicts source, invents specs, or omits Name/Material/Warranty.",
}

def judge_single_criterion(row: pd.Series, criterion_name: str, client: OpenAI, model: str = JUDGE_MODEL) -> dict:
    rubric = CRITERION_RUBRICS[criterion_name]
    extra = "\nIMPORTANT: Use careful step-by-step counting. List each word numbered before stating the total.\n" if criterion_name == "length" else (
            "\nIMPORTANT: Compare description against SOURCE DATA carefully.\n" if criterion_name == "grounding" else "")
    system = (f"You are an expert product-copy evaluator evaluating ONE criterion only.\n\n{rubric}{extra}\n"
              "Write explanation first, then verdict (good/ok/bad). JSON only.")
    t0 = time.time()
    response = client.chat.completions.create(
        model=model, temperature=0.3,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": build_user_message(row)}],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "SingleCriterionResult", "schema": SingleCriterionResult.model_json_schema()},
        },
    )
    latency_ms = (time.time() - t0) * 1000
    parsed = json.loads(response.choices[0].message.content)
    return {"explanation": parsed["explanation"], "verdict": parsed["verdict"],
            "judge_latency_ms": round(latency_ms, 1),
            "judge_input_tokens": response.usage.prompt_tokens,
            "judge_output_tokens": response.usage.completion_tokens}

# ── Pass/fail ─────────────────────────────────────────────────────────────
VERDICT_TO_SCORE = {"good": 3, "ok": 2, "bad": 1}
JUDGE_CRITERIA   = ["fluency", "grammar", "tone", "length", "grounding"]
ALL_CRITERIA     = JUDGE_CRITERIA + ["latency_rating", "cost_rating"]

def compute_pass_fail(row: pd.Series, verdict_suffix: str = "_verdict") -> dict:
    ratings = {c: row.get(f"{c}{verdict_suffix}", "bad") for c in JUDGE_CRITERIA}
    ratings["latency_rating"] = row.get("latency_rating", "bad")
    ratings["cost_rating"]    = row.get("cost_rating",    "bad")
    total_score = sum(VERDICT_TO_SCORE.get(v, 1) for v in ratings.values())
    num_good    = sum(1 for v in ratings.values() if v == "good")
    auto_fail   = ratings.get("grounding") == "bad" or ratings.get("length") == "bad" or ratings.get("grammar") == "bad"
    final = "fail" if auto_fail else ("pass" if num_good >= 5 and total_score >= 17 else "fail")
    return {"total_score": total_score, "num_good": num_good, "final_verdict": final}

print("✓ Session restored — client, data, schemas, functions all ready.")
print(f"  df: {len(df)} products | df_human: {len(df_human)} products")
print(f"  JUDGE_MODEL    = {JUDGE_MODEL}")
print(f"  JUDGE_MODEL_V2 = {JUDGE_MODEL_V2}")

✓ Session restored — client, data, schemas, functions all ready.
  df: 50 products | df_human: 50 products
  JUDGE_MODEL    = meta-llama/Meta-Llama-3.1-8B-Instruct
  JUDGE_MODEL_V2 = Qwen/Qwen3-30B-A3B-Instruct-2507


---

# Task 6: Run the Judge and Analyze Results

## 6.1 Sanity Check — Judge the First 5 Products

Run the all-criteria judge on a small sample and manually review the
explanations and verdicts to make sure the judge is behaving sensibly.

In [144]:
# ── Sanity check: judge first 5 products ────────────────────────────────
sanity_results = []

for i in range(min(5, len(df))):
    row = df.iloc[i]
    print(f"[{i+1}/5] Judging: {row['product_name']}")
    try:
        result = judge_all_criteria(row, client)
        result["product_name"] = row["product_name"]
        sanity_results.append(result)
    except Exception as e:
        print(f"  ERROR: {e}")
        sanity_results.append({"product_name": row["product_name"], "error": str(e)})
    time.sleep(0.5)

print("\nDone.")

[1/5] Judging: Apple iPhone 15 Pro
[2/5] Judging: Samsung Galaxy S24 Ultra
[3/5] Judging: Google Pixel 8 Pro
[4/5] Judging: Sony WH‑1000XM5 Headphones
[5/5] Judging: Bose QuietComfort Ultra Earbuds

Done.


In [145]:
# ── Display sanity check results ────────────────────────────────────────
for r in sanity_results:
    if "error" in r:
        print(f"\n{'='*60}")
        print(f"PRODUCT: {r['product_name']}")
        print(f"ERROR: {r['error']}")
        continue

    print(f"\n{'='*60}")
    print(f"PRODUCT: {r['product_name']}")
    print(f"  Latency: {r['judge_latency_ms']:.0f} ms | "
          f"Tokens: {r['judge_input_tokens']}in/{r['judge_output_tokens']}out")
    for c in JUDGE_CRITERIA:
        print(f"\n  [{c.upper()}] verdict={r[f'{c}_verdict']}")
        print(f"    {r[f'{c}_explanation'][:200]}")


PRODUCT: Apple iPhone 15 Pro
  Latency: 8960 ms | Tokens: 1051in/352out

  [FLUENCY] verdict=ok
    The description uses a mix of short and long sentences, but the transition between the second and third sentences feels a bit abrupt. Additionally, the first sentence starts with 'Experience,' and the

  [GRAMMAR] verdict=good
    There are no spelling, punctuation, or grammatical errors in the description.

  [TONE] verdict=ok
    The description uses a sales-oriented tone, but it relies on only one persuasive technique, which is benefit framing. It does not use any emotional or aspirational language, and it does not create a s

  [LENGTH] verdict=ok
    Here are the words in the description, numbered for accuracy: 1:Experience 2:the 3:power 4:and 5:precision 6:of 7:the 8:Apple 9:iPhone 10:15 11:Pro.  There are 11 words. The description is within the 

  [GROUNDING] verdict=ok
    The description mentions the Product Name (Apple iPhone 15 Pro), Material (titanium frame and Ceramic Shie

## 6.1.5 Spot Check — Judge vs Human Evaluation (Task 3 Baseline)

Cross-reference the sanity-check results (first 5 products) against the
human ratings recorded in `task_03_gemma.xlsx`.  For each product and
criterion, print whether the judge agreed or disagreed with the human rater.

Pay particular attention to the **Bose QuietComfort Ultra Earbuds** row:
the LLM judge reported the description contains **76 words, which it claimed
falls outside the acceptable 55–85 word range** and downgraded `length` to
`bad`.  This is a double hallucination:

1. **Wrong count** — `len(description.split())` gives **77 words**, not 76.
2. **Wrong verdict** — Even worse !! 76 is squarely **inside** the 55–85
   word target; the correct rating is `good`.



In [146]:
# ── Load Task-3 human baseline for spot-check comparison ────────────────────
task3_path = "task_03_gemma.xlsx"
df_human = pd.read_excel(task3_path)

# Keep only the 5 products that were sanity-checked
sanity_names = [r["product_name"] for r in sanity_results if "error" not in r]
df_sanity_human = df_human[df_human["product_name"].isin(sanity_names)].set_index("product_name")

print(f"Human-rated products available for spot-check: {len(df_sanity_human)}")
print()

# ── Side-by-side comparison ──────────────────────────────────────────────────
agree_total = 0
compare_total = 0

for r in sanity_results:
    if "error" in r:
        continue
    name = r["product_name"]
    if name not in df_sanity_human.index:
        print(f"{name[:40]}: no human eval available")
        continue

    human_row = df_sanity_human.loc[name]
    actual_wc = len(str(human_row.get("generated_description", "")).split())

    print(f"\n{'='*64}")
    print(f"PRODUCT : {name}")
    print(f"Description word count (counted in code): {actual_wc}")
    print(f"  {'Criterion':12s}  {'Human':>6s}  {'Judge':>6s}  {'Match':>5s}  Notes")
    print(f"  {'-'*60}")

    for c in JUDGE_CRITERIA:
        human_rating = human_row.get(c, None)
        judge_rating = r.get(f"{c}_verdict", None)

        if pd.isna(human_rating) if human_rating is not None else True:
            match_flag = " — "
            note = "no human rating"
        else:
            compare_total += 1
            if human_rating == judge_rating:
                agree_total += 1
                match_flag = "✓"
                note = ""
            else:
                match_flag = "✗"
                note = ""

        print(f"  {c:12s}  {str(human_rating):>6s}  {str(judge_rating):>6s}  {match_flag:>5s}  {note}")

        # ── Bose length hallucination callout ────────────────────────────────
        if "Bose" in name and c == "length":
            judge_expl = r.get("length_explanation", "")
            print(f"")
            print(f"  *** HALLUCINATION ALERT ***")
            print(f"  Judge explanation: \"{judge_expl[:150]}\"")
            print(f"  Problem: judge claims the description has 76 words and is")
            print(f"  OUTSIDE the 55–85 range — but the actual word count is {actual_wc},")
            print(f"  which is clearly inside 55–85.  Human correctly rated this 'good'.")
            print(f"  The model fabricated a specific number and then reasoned from it.")
            print()

print(f"\n{'='*64}")
if compare_total > 0:
    print(f"Spot-check agreement: {agree_total}/{compare_total} ({agree_total/compare_total:.0%})")
else:
    print("No comparable human ratings found in this sanity sample.")
print()


Human-rated products available for spot-check: 5


PRODUCT : Apple iPhone 15 Pro
Description word count (counted in code): 65
  Criterion      Human   Judge  Match  Notes
  ------------------------------------------------------------
  fluency         good      ok      ✗  
  grammar         good    good      ✓  
  tone            good      ok      ✗  
  length          good      ok      ✗  
  grounding       good      ok      ✗  

PRODUCT : Samsung Galaxy S24 Ultra
Description word count (counted in code): 98
  Criterion      Human   Judge  Match  Notes
  ------------------------------------------------------------
  fluency         good      ok      ✗  
  grammar         good    good      ✓  
  tone            good      ok      ✗  
  length           bad      ok      ✗  
  grounding       good      ok      ✗  

PRODUCT : Google Pixel 8 Pro
Description word count (counted in code): 68
  Criterion      Human   Judge  Match  Notes
  -------------------------------------------------------

## 6.1.6 Upgrade to a Larger Judge Model — Qwen3-30B-A3B-Instruct-2507

> **To run this section after a kernel restart**, run **cell 6.0 (Restore Session)** first — it recreates `client`, `df`, `df_human`, all constants, and all functions in one shot.

### Why switch?

The`meta-llama/Meta-Llama-3.1-8B-Instruct` judge agrred with the human(us) only on 24% of the results. 
Even worse is the **Bose QuietComfort Ultra Earbuds**  example, it hallucinated a word count (76 words) and then
incorrectly decided 76 is out of the 55–85 word range, giving `bad`
for `length` while the human rater gave `good` — and `split()` count
confirms the description has **77 words**.

Looks like the model struggle on **grounded, quantitative reasoning** (counting, range checks).

### Why Qwen3-30B-A3B-Instruct-2507?

* **30 B parameters** — substantially larger reasoning capacity for grounded
  checks like word-count verification and attribute cross-referencing
* **Strong instruction-following** — Qwen3 models score well on structured-
  output benchmarks and are less prone to hallucinating measurable quantities


### What stays the same

`judge_all_criteria()` and `judge_single_criterion()` both accept a `model=`
argument, so no prompt or schema changes are needed — we just pass the new
model ID.

In [147]:
# ── Switch to the larger judge model ────────────────────────────────────────
JUDGE_MODEL_V2 = "Qwen/Qwen3-30B-A3B-Instruct-2507"
print(f"Upgraded judge model: {JUDGE_MODEL_V2}")
print(f"Previous judge model: {JUDGE_MODEL}")
print

Upgraded judge model: Qwen/Qwen3-30B-A3B-Instruct-2507
Previous judge model: meta-llama/Meta-Llama-3.1-8B-Instruct


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [148]:
# ── Sanity check with upgraded model: first 5 products ──────────────────────
sanity_results_v2 = []

for i in range(min(5, len(df))):
    row = df.iloc[i]
    print(f"[{i+1}/5] Judging (v2): {row['product_name']}")
    try:
        result = judge_all_criteria(row, client, model=JUDGE_MODEL_V2)
        result["product_name"] = row["product_name"]
        sanity_results_v2.append(result)
    except Exception as e:
        print(f"  ERROR: {e}")
        sanity_results_v2.append({"product_name": row["product_name"], "error": str(e)})
    time.sleep(0.5)

print("\nDone.")

[1/5] Judging (v2): Apple iPhone 15 Pro
[2/5] Judging (v2): Samsung Galaxy S24 Ultra
[3/5] Judging (v2): Google Pixel 8 Pro
[4/5] Judging (v2): Sony WH‑1000XM5 Headphones
[5/5] Judging (v2): Bose QuietComfort Ultra Earbuds

Done.


In [149]:
# ── Compare Qwen3 judge results to human evaluation (task_03_gemma) ──────────
# Note: word counts are computed from `df` (assignment_01.xlsx) 
agree_total_v2 = 0
compare_total_v2 = 0

sanity_names_v2   = [r["product_name"] for r in sanity_results_v2 if "error" not in r]
df_judged         = df.set_index("product_name")          # descriptions judge saw
df_sanity_human   = df_human[df_human["product_name"].isin(sanity_names_v2)].set_index("product_name")

print(f"Human-rated products available for v2 spot-check: {len(df_sanity_human)}")
print()

for r in sanity_results_v2:
    if "error" in r:
        print(f"\n{r['product_name']}: ERROR — {r['error']}")
        continue

    name = r["product_name"]
    if name not in df_sanity_human.index:
        print(f"{name[:40]}: no human eval available")
        continue

    human_row  = df_sanity_human.loc[name]
    judged_row = df_judged.loc[name]
    actual_wc  = len(str(judged_row.get("generated_description", "")).split())

    print(f"\n{'='*64}")
    print(f"PRODUCT : {name}")
    print(f"Actual word count (counted by Python code): {actual_wc}")
    print(f"  {'Criterion':12s}  {'Human':>6s}  {'Qwen3':>6s}  {'Match':>5s}  Notes")
    print(f"  {'-'*60}")

    for c in JUDGE_CRITERIA:
        human_rating = human_row.get(c, None)
        judge_rating = r.get(f"{c}_verdict", None)

        if pd.isna(human_rating) if human_rating is not None else True:
            match_flag = " — "
            note = "no human rating"
        else:
            compare_total_v2 += 1
            if human_rating == judge_rating:
                agree_total_v2 += 1
                match_flag = "✓"
                note = ""
            else:
                match_flag = "✗"
                note = ""

        print(f"  {c:12s}  {str(human_rating):>6s}  {str(judge_rating):>6s}  {match_flag:>5s}  {note}")

        # Bose length callout — did Qwen3 fix the hallucination?
        if "Bose" in name and c == "length":
            judge_expl = r.get("length_explanation", "")
            fixed  = (judge_rating == human_rating)
            status = "FIXED ✓" if fixed else "STILL WRONG ✗"
            print(f"")
            print(f"  *** Bose length check ({status}) ***")
            print(f"  Qwen3 explanation: \"{judge_expl[:200]}\"")
            print(f"  Actual word count (counted by Python code): {actual_wc} (inside 55–85: {55 <= actual_wc <= 85})")
            print()

print(f"\n{'='*64}")
if compare_total_v2 > 0:
    print(f"Qwen3 vs human agreement: {agree_total_v2}/{compare_total_v2} ({agree_total_v2/compare_total_v2:.0%})")
else:
    print("No comparable human ratings found in this sample.")

# ── Detailed explanations ────────────────────────────────────────────────────
print(f"\n{'='*64}")
print("DETAILED Qwen3 EXPLANATIONS")
print(f"{'='*64}")
for r in sanity_results_v2:
    if "error" in r:
        continue
    name = r["product_name"]
    judged_desc = df_judged.loc[name, "generated_description"] if name in df_judged.index else "N/A"
    actual_wc   = len(str(judged_desc).split())
    print(f"\n{'='*64}")
    print(f"PRODUCT: {name}  (actual word count, counted by Python code: {actual_wc})")
    print(f"  Latency: {r['judge_latency_ms']:.0f} ms | "
          f"Tokens: {r['judge_input_tokens']}in/{r['judge_output_tokens']}out")
    for c in JUDGE_CRITERIA:
        print(f"\n  [{c.upper()}] verdict={r[f'{c}_verdict']}")
        print(f"    {r[f'{c}_explanation'][:250]}")

Human-rated products available for v2 spot-check: 5


PRODUCT : Apple iPhone 15 Pro
Actual word count (counted by Python code): 65
  Criterion      Human   Qwen3  Match  Notes
  ------------------------------------------------------------
  fluency         good    good      ✓  
  grammar         good    good      ✓  
  tone            good    good      ✓  
  length          good      ok      ✗  
  grounding       good    good      ✓  

PRODUCT : Samsung Galaxy S24 Ultra
Actual word count (counted by Python code): 98
  Criterion      Human   Qwen3  Match  Notes
  ------------------------------------------------------------
  fluency         good    good      ✓  
  grammar         good    good      ✓  
  tone            good    good      ✓  
  length           bad     bad      ✓  
  grounding       good    good      ✓  

PRODUCT : Google Pixel 8 Pro
Actual word count (counted by Python code): 68
  Criterion      Human   Qwen3  Match  Notes
  ----------------------------------------------

## intermidate conclusions
I strated with temprature 0.0 this resulted with the Model can't judge on word count, it gave the same generic explantion for fluency and grammar.
changing to tempature to 0.3 gave good results:

1. **Length**: The judge now follows the chain-of-thought counting instruction
   (listing words numbered). At `temperature=0.0` it ignored this instruction
   entirely and defaulted to "78 words" for every product.
2. **Grounding**: The judge caught the missing `capacity: large` attribute on
   Sony that it missed before.
3. **Explanation quality**: No longer templated. Each product gets specific
   observations rather than copy-pasted boilerplate.
4. **Samsung length fixed**: The most egregious error (98 words rated "good")
   is now correctly rated "bad."

## Summary

| Criterion | Agreement | Explanation quality | Notes |
|-----------|-----------|-------------------|-------|
| Fluency   | 5/5 (100%) | Product-specific | Improved from generic |
| Grammar   | 5/5 (100%) | Product-specific | Improved from generic |
| Tone      | 5/5 (100%) | Strong — cites real evidence | Best criterion |
| Grounding | 4/5 (80%)  | Improved | Still misses Pixel rating omission |
| Length    | 4/5 (80%)  | Chain-of-thought counting | Major improvement |

Overall agreement: 23/25 (92%)

The judge is now reliable for all subjective criteria and significantly
better at length counting via chain-of-thought enumeration. The remaining
weaknesses are:
1. One grounding miss (Pixel 8 Pro rating attribute)
2. One length miss (Apple iPhone 15 Pro off by one tier)



## 6.2 Full Run — All Criteria at Once

Run the judge on every product in the dataset. Save results with pass/fail
logic to `task_06_all_criteria.xlsx`.

In [150]:
# ── Full run: all-criteria-at-once ───────────────────────────────────────
all_criteria_results = []

for idx, row in df.iterrows():
    print(f"[{idx+1}/{len(df)}] Judging: {row['product_name']}")
    try:
        result = judge_all_criteria(row, client)
    except Exception as e:
        print(f"  ERROR: {e}")
        result = {}
        for c in JUDGE_CRITERIA:
            result[f"{c}_explanation"] = f"ERROR: {e}"
            result[f"{c}_verdict"] = "bad"
        result["judge_latency_ms"] = 0
        result["judge_input_tokens"] = 0
        result["judge_output_tokens"] = 0
    all_criteria_results.append(result)
    time.sleep(0.5)

print(f"\nCompleted judging {len(all_criteria_results)} products.")

[1/50] Judging: Apple iPhone 15 Pro
[2/50] Judging: Samsung Galaxy S24 Ultra
[3/50] Judging: Google Pixel 8 Pro
[4/50] Judging: Sony WH‑1000XM5 Headphones
[5/50] Judging: Bose QuietComfort Ultra Earbuds
[6/50] Judging: Amazon Echo Dot (5th Gen)
[7/50] Judging: Dell XPS 13 9310 Laptop
[8/50] Judging: Apple MacBook Air 13″ (M3)
[9/50] Judging: Microsoft Surface Pro 10
[10/50] Judging: Garmin Forerunner 255
[11/50] Judging: Fitbit Charge 6
[12/50] Judging: GoPro HERO12 Black
[13/50] Judging: DJI Mini 4 Pro Drone
[14/50] Judging: Nintendo Switch OLED
[15/50] Judging: PlayStation 5 Slim
[16/50] Judging: Xbox Series X
[17/50] Judging: Instant Pot Duo 6‑Quart
[18/50] Judging: Keurig K‑Supreme Plus Smart
[19/50] Judging: Vitamix 5200 Blender
[20/50] Judging: Dyson V15 Detect Vacuum
[21/50] Judging: iRobot Roomba j7+
[22/50] Judging: Yeti Rambler 20 oz Tumbler
[23/50] Judging: Stanley Quencher H2.0 40 oz
[24/50] Judging: Hydro Flask 32 oz Wide Mouth
[25/50] Judging: Contigo Autoseal West Loop 1

In [169]:
import re

# ── Build results DataFrame and apply pass/fail ─────────────────────────
judge_df = df.copy()
results_df = pd.DataFrame(all_criteria_results)

# Add judge columns
for col in results_df.columns:
    judge_df[col] = results_df[col].values

# Apply pass/fail logic
pf_results = judge_df.apply(compute_pass_fail, axis=1, result_type="expand")
judge_df["judge_total_score"] = pf_results["total_score"]
judge_df["judge_num_good"] = pf_results["num_good"]
judge_df["judge_final_verdict"] = pf_results["final_verdict"]

# ── Clean up the verbose length explanation ─────────────────────────────
def clean_length_text(text):
    if not isinstance(text, str): 
        return text
    match = re.search(r'(\d+)\s*words', text, re.IGNORECASE)
    num = match.group(1) if match else "??"
    return f"The product description has {num} words, counted one by one."

if "length_explanation" in judge_df.columns:
    judge_df["length_explanation"] = judge_df["length_explanation"].apply(clean_length_text)

# ── Format and Reorder Columns specifically for Excel Export ────────────
# We create a copy so we don't break the comparison logic in cell 6.3
export_judge_df = judge_df.copy()

# 1. Overwrite the empty criteria columns with the verdict values, then drop the _verdict columns
for c in JUDGE_CRITERIA:
    export_judge_df[c] = export_judge_df[f"{c}_verdict"]
    export_judge_df.drop(columns=[f"{c}_verdict"], inplace=True)

# 2. Put the pass/fail value into 'final_score' and drop the old tracking columns
export_judge_df["final_score"] = export_judge_df["judge_final_verdict"]
export_judge_df.drop(columns=["judge_total_score", "judge_num_good", "judge_final_verdict"], inplace=True, errors="ignore")

# 3. Define the exact order requested
core_cols = [
    "product_name", "Product_attribute_list", "material", "warranty", 
    "generated_description", "latency_ms", "input_tokens", "output_tokens"
]
core_cols  = [c for c in core_cols if c in export_judge_df.columns] # Ensure they exist

verdicts   = JUDGE_CRITERIA # fluency, grammar, tone, length, grounding
score_cols = [c for c in ["latency_rating", "cost_rating", "final_score"] if c in export_judge_df.columns]
expl_cols  = [f"{c}_explanation" for c in JUDGE_CRITERIA]

# 4. Gather all remaining columns and apply the new order
used_cols  = set(core_cols + verdicts + score_cols + expl_cols)
other_cols = [col for col in export_judge_df.columns if col not in used_cols]

final_col_order = core_cols + verdicts + score_cols + expl_cols + other_cols
export_judge_df = export_judge_df[final_col_order]

# ── Save ────────────────────────────────────────────────────────────────
full_run_output_path = "task_06_full_run.xlsx"
export_judge_df.to_excel(full_run_output_path, index=False, engine="openpyxl")
print(f"Saved {len(export_judge_df)} rows to {full_run_output_path}")

# Summary
print("\n=== All-Criteria Judge Summary ===")
print(f"Total products: {len(judge_df)}")
print(f"Pass: {(judge_df['judge_final_verdict'] == 'pass').sum()}")
print(f"Fail: {(judge_df['judge_final_verdict'] == 'fail').sum()}")
print(f"\nVerdict distribution per criterion:")
for c in JUDGE_CRITERIA:
    col = f"{c}_verdict"
    counts = judge_df[col].value_counts()
    print(f"  {c:12s}  good={counts.get('good',0):2d}  ok={counts.get('ok',0):2d}  bad={counts.get('bad',0):2d}")

Saved 50 rows to task_06_full_run.xlsx

=== All-Criteria Judge Summary ===
Total products: 50
Pass: 0
Fail: 50

Verdict distribution per criterion:
  fluency       good= 1  ok=49  bad= 0
  grammar       good=45  ok= 5  bad= 0
  tone          good=12  ok=38  bad= 0
  length        good= 8  ok= 7  bad=35
  grounding     good= 3  ok=30  bad=17


### Note! Full run results are not as good as 5 products they result in fail criteria.

## 6.3 Compare Judge to Human Evaluation

Only products that received human evaluation in Task 3 can be compared.
For each of the 5 criteria, compute the agreement rate between the judge
verdict and the human rating.

In [171]:
def compare_judge_to_human(
    data: pd.DataFrame,
    verdict_suffix: str = "_verdict",
    label: str = "all-at-once",
) -> pd.DataFrame:
    """
    Compare judge verdicts to human ratings for products that have human eval.

    Returns a DataFrame with per-criterion agreement rates.
    """
    # Load human evaluations from Task 3
    df_human_local = pd.read_excel("task_03_gemma.xlsx")
    
    # Extract only the columns we need from human eval
    human_cols = ["product_name"] + JUDGE_CRITERIA
    human_data = df_human_local[human_cols].copy()
    
    # Ensure product_name is string and stripped of whitespace to match perfectly
    data_copy = data.copy()
    data_copy["product_name"] = data_copy["product_name"].astype(str).str.strip()
    human_data["product_name"] = human_data["product_name"].astype(str).str.strip()

    # Drop existing empty human criteria from 'data' to avoid merge conflicts
    judge_data_clean = data_copy.drop(columns=JUDGE_CRITERIA, errors="ignore")
    
    # Merge judge data with human data
    merged = pd.merge(
        judge_data_clean, 
        human_data, 
        on="product_name", 
        how="left"
    )

    # Filter to rows that have human evaluation
    has_human = merged["fluency"].notna()
    subset = merged[has_human].copy()
    n = len(subset)
    print(f"Comparing {n} products with human evaluation ({label} mode)\n")

    rows = []
    for c in JUDGE_CRITERIA:
        human_col = c  # human ratings are in columns like 'fluency', 'grammar', etc.
        judge_col = f"{c}{verdict_suffix}"

        if judge_col not in subset.columns:
            print(f"  WARNING: {judge_col} not found in data, skipping.")
            continue

        matches = (subset[human_col] == subset[judge_col]).sum()
        agreement = matches / n if n > 0 else 0

        # Detailed breakdown
        details = []
        for _, row in subset.iterrows():
            h = row[human_col]
            j = row[judge_col]
            if h != j:
                details.append(f"{row['product_name'][:25]}: human={h} vs judge={j}")

        rows.append({
            "criterion": c,
            "agreement": f"{agreement:.0%}",
            "matches": matches,
            "total": n,
            "disagreements": "; ".join(details[:5]) if details else "none",
        })

        print(f"  {c:12s}  {matches}/{n} agree ({agreement:.0%})")

    return pd.DataFrame(rows)


# ── Run comparison for all-at-once mode ─────────────────────────────────
agreement_all = compare_judge_to_human(judge_df, "_verdict", "all-at-once")
agreement_all

Comparing 10 products with human evaluation (all-at-once mode)

  fluency       0/10 agree (0%)
  grammar       9/10 agree (90%)
  tone          3/10 agree (30%)
  length        0/10 agree (0%)
  grounding     5/10 agree (50%)


,criterion,agreement,matches,total,disagreements
0,fluency,0%,0,10,Apple iPhone 15 Pro: human=good vs judge=ok; S...
1,grammar,90%,9,10,Microsoft Surface Pro 10: human=good vs judge=ok
2,tone,30%,3,10,Apple iPhone 15 Pro: human=good vs judge=ok; G...
3,length,0%,0,10,Apple iPhone 15 Pro: human=good vs judge=bad; ...
4,grounding,50%,5,10,Apple iPhone 15 Pro: human=good vs judge=bad; ...


## 6.4 Criterion-by-Criterion Judging

Instead of evaluating all 5 criteria in one API call, evaluate each criterion
in its own dedicated call. This gives the model a simpler, more focused task
and may improve accuracy — at the cost of 5x more API calls.

In [172]:
# ── Criterion-by-criterion full run ─────────────────────────────────────
cbc_results = []  # one dict per product
MANAUAL_EVALUATED_PRODUCTS = 10  # set to len(df) to run all, or a smaller number for testing
#total_calls = len(df) * len(JUDGE_CRITERIA)
total_calls = MANAUAL_EVALUATED_PRODUCTS * len(JUDGE_CRITERIA)
call_num = 0

for idx, row in df.head(10).iterrows():
    product_result = {}
    total_latency = 0
    total_input_tokens = 0
    total_output_tokens = 0

    for criterion in JUDGE_CRITERIA:
        call_num += 1
        print(f"[{call_num}/{total_calls}] {row['product_name'][:30]} — {criterion}")
        try:
            r = judge_single_criterion(row, criterion, client)
            product_result[f"{criterion}_explanation"] = r["explanation"]
            product_result[f"{criterion}_verdict"] = r["verdict"]
            total_latency += r["judge_latency_ms"]
            total_input_tokens += r["judge_input_tokens"]
            total_output_tokens += r["judge_output_tokens"]
        except Exception as e:
            print(f"  ERROR: {e}")
            product_result[f"{criterion}_explanation"] = f"ERROR: {e}"
            product_result[f"{criterion}_verdict"] = "bad"
        time.sleep(0.5)

    product_result["judge_latency_ms"] = round(total_latency, 1)
    product_result["judge_input_tokens"] = total_input_tokens
    product_result["judge_output_tokens"] = total_output_tokens
    cbc_results.append(product_result)

print(f"\nCompleted criterion-by-criterion judging: {len(cbc_results)} products.")

[1/50] Apple iPhone 15 Pro — fluency
[2/50] Apple iPhone 15 Pro — grammar
[3/50] Apple iPhone 15 Pro — tone
[4/50] Apple iPhone 15 Pro — length
[5/50] Apple iPhone 15 Pro — grounding
[6/50] Samsung Galaxy S24 Ultra — fluency
[7/50] Samsung Galaxy S24 Ultra — grammar
[8/50] Samsung Galaxy S24 Ultra — tone
[9/50] Samsung Galaxy S24 Ultra — length
[10/50] Samsung Galaxy S24 Ultra — grounding
[11/50] Google Pixel 8 Pro — fluency
[12/50] Google Pixel 8 Pro — grammar
[13/50] Google Pixel 8 Pro — tone
[14/50] Google Pixel 8 Pro — length
[15/50] Google Pixel 8 Pro — grounding
[16/50] Sony WH‑1000XM5 Headphones — fluency
[17/50] Sony WH‑1000XM5 Headphones — grammar
[18/50] Sony WH‑1000XM5 Headphones — tone
[19/50] Sony WH‑1000XM5 Headphones — length
[20/50] Sony WH‑1000XM5 Headphones — grounding
[21/50] Bose QuietComfort Ultra Earbud — fluency
[22/50] Bose QuietComfort Ultra Earbud — grammar
[23/50] Bose QuietComfort Ultra Earbud — tone
[24/50] Bose QuietComfort Ultra Earbud — length
[25/50] Bo

In [173]:
import re

# ── Build criterion-by-criterion DataFrame and apply pass/fail ──────────
# Match the size of the dataframe to the number of products you actually judged
cbc_df = df.head(MANAUAL_EVALUATED_PRODUCTS).copy()
cbc_results_df = pd.DataFrame(cbc_results)

for col in cbc_results_df.columns:
    cbc_df[col] = cbc_results_df[col].values

# Apply pass/fail
pf_cbc = cbc_df.apply(compute_pass_fail, axis=1, result_type="expand")
cbc_df["judge_total_score"] = pf_cbc["total_score"]
cbc_df["judge_num_good"] = pf_cbc["num_good"]
cbc_df["judge_final_verdict"] = pf_cbc["final_verdict"]

# ── Clean up the verbose length explanation ─────────────────────────────
def clean_length_text(text):
    if not isinstance(text, str): 
        return text
    match = re.search(r'(\d+)\s*words', text, re.IGNORECASE)
    num = match.group(1) if match else "??"
    return f"The product description has {num} words, counted one by one."

if "length_explanation" in cbc_df.columns:
    cbc_df["length_explanation"] = cbc_df["length_explanation"].apply(clean_length_text)

# ── Format and Reorder Columns specifically for Excel Export ────────────
# We create a copy so we don't break the comparison logic in cell 6.5
export_df = cbc_df.copy()

# 1. Overwrite the empty criteria columns with the verdict values, then drop the _verdict columns
for c in JUDGE_CRITERIA:
    export_df[c] = export_df[f"{c}_verdict"]
    export_df.drop(columns=[f"{c}_verdict"], inplace=True)

# 2. Put the pass/fail value into 'final_score' and drop the old tracking columns
export_df["final_score"] = export_df["judge_final_verdict"]
export_df.drop(columns=["judge_total_score", "judge_num_good", "judge_final_verdict"], inplace=True, errors="ignore")

# 3. Define the exact order requested
core_cols = [
    "product_name", "Product_attribute_list", "material", "warranty", 
    "generated_description", "latency_ms", "input_tokens", "output_tokens"
]
core_cols  = [c for c in core_cols if c in export_df.columns] # Ensure they exist

verdicts   = JUDGE_CRITERIA # fluency, grammar, tone, length, grounding
score_cols = [c for c in ["latency_rating", "cost_rating", "final_score"] if c in export_df.columns]
expl_cols  = [f"{c}_explanation" for c in JUDGE_CRITERIA]

# 4. Gather all remaining columns and apply the new order
used_cols  = set(core_cols + verdicts + score_cols + expl_cols)
other_cols = [col for col in export_df.columns if col not in used_cols]

final_col_order = core_cols + verdicts + score_cols + expl_cols + other_cols
export_df = export_df[final_col_order]

# ── Save ────────────────────────────────────────────────────────────────
cbc_output_path = "task_06_product_criteria.xlsx"
export_df.to_excel(cbc_output_path, index=False, engine="openpyxl")
print(f"Saved {len(export_df)} rows to {cbc_output_path}")

# Summary
print(f"\n=== Criterion-by-Criterion Judge Summary ===")
print(f"Pass: {(cbc_df['judge_final_verdict'] == 'pass').sum()}")
print(f"Fail: {(cbc_df['judge_final_verdict'] == 'fail').sum()}")
print(f"\nVerdict distribution per criterion:")
for c in JUDGE_CRITERIA:
    col = f"{c}_verdict"
    counts = cbc_df[col].value_counts()
    print(f"  {c:12s}  good={counts.get('good',0):2d}  ok={counts.get('ok',0):2d}  bad={counts.get('bad',0):2d}")

Saved 10 rows to task_06_product_criteria.xlsx

=== Criterion-by-Criterion Judge Summary ===
Pass: 0
Fail: 10

Verdict distribution per criterion:
  fluency       good= 6  ok= 4  bad= 0
  grammar       good= 1  ok= 9  bad= 0
  tone          good=10  ok= 0  bad= 0
  length        good= 0  ok= 0  bad=10
  grounding     good= 6  ok= 4  bad= 0


## 6.5 Compare Both Judging Modes to Human Evaluation

Side-by-side comparison: did evaluating each criterion in isolation improve
agreement with human ratings?

In [174]:
# ── Compare criterion-by-criterion to human ─────────────────────────────
print("=" * 60)
print("CRITERION-BY-CRITERION vs HUMAN")
print("=" * 60)
agreement_cbc = compare_judge_to_human(cbc_df, "_verdict", "criterion-by-criterion")

print("\n")
print("=" * 60)
print("ALL-AT-ONCE vs HUMAN (repeated for comparison)")
print("=" * 60)
agreement_all_repeat = compare_judge_to_human(judge_df.head(MANAUAL_EVALUATED_PRODUCTS), "_verdict", "all-at-once")

# ── Side-by-side summary table ──────────────────────────────────────────
print("\n\n=== SIDE-BY-SIDE COMPARISON ===\n")
print(f"{'Criterion':12s}  {'All-at-once':>12s}  {'Crit-by-crit':>12s}")
print("-" * 42)
for c in JUDGE_CRITERIA:
    all_row = agreement_all_repeat[agreement_all_repeat["criterion"] == c]
    cbc_row = agreement_cbc[agreement_cbc["criterion"] == c]
    all_val = all_row["agreement"].values[0] if len(all_row) > 0 else "N/A"
    cbc_val = cbc_row["agreement"].values[0] if len(cbc_row) > 0 else "N/A"
    print(f"  {c:12s}  {all_val:>12s}  {cbc_val:>12s}")

CRITERION-BY-CRITERION vs HUMAN
Comparing 10 products with human evaluation (criterion-by-criterion mode)

  fluency       6/10 agree (60%)
  grammar       1/10 agree (10%)
  tone          10/10 agree (100%)
  length        2/10 agree (20%)
  grounding     3/10 agree (30%)


ALL-AT-ONCE vs HUMAN (repeated for comparison)
Comparing 10 products with human evaluation (all-at-once mode)

  fluency       0/10 agree (0%)
  grammar       9/10 agree (90%)
  tone          3/10 agree (30%)
  length        0/10 agree (0%)
  grounding     5/10 agree (50%)


=== SIDE-BY-SIDE COMPARISON ===

Criterion      All-at-once  Crit-by-crit
------------------------------------------
  fluency                 0%           60%
  grammar                90%           10%
  tone                   30%          100%
  length                  0%           20%
  grounding              50%           30%


## 6.6 Analysis

### Why might criterion-by-criterion judging produce different results?

When evaluating all criteria at once, the judge's assessment of one criterion can **bleed into another**. 
For example, if the model notices a grounding issue, it may become more critical of tone or fluency in the same pass. Isolating criteria forces the judge to focus on one dimension at a time, reducing cross-criterion contamination.

Additionally, with a single focused rubric, the smaller model has less instruction to process, resultoing in smaller context, and what might improve adherence to the specific criterion definitions.

The-all-at-once did perform better on grammar.

### Trade-offs: Human Evaluation vs LLM-as-Judge

| Dimension | Human Evaluation | LLM-as-Judge |
|---|---|---|
| **Cost** | High (reviewer time) | Low (API calls ~$0.0001 each) |
| **Scale** | 10--15 products feasible | 1000s of products in minutes |
| **Consistency** | Varies with fatigue, mood | Might Hallucinate, have to be find tuned espcially in smaller models |
| **Accuracy** | Gold standard for subjective criteria | Good for clear-cut criteria, weaker on nuance |
| **Grounding** | Humans catch subtle hallucinations | Models may miss plausible-sounding inventions |
| **Speed** | Minutes per product | Seconds per product |

### Recommendation for Production

For a system generating thousands of descriptions daily, use a **tiered approach**:
1. **Chossing model, fine tuning agent's param** 
Strating with a few products, identifying mismatch versus human evaluation, choosing the model, and fine tuning the agent params like prompt, temprature and top_p can have meaningfull positive impact. 
2. **Critical accuracy** If you have criteria's which must be accurtae you must carefully test your agent and consider doing them by code if agent doesn't align.
3. **LLM judge on 100%** of descriptions as a first-pass filter (fast, cheap, consistent)
4. **Human review on failures** and a random sample of passes (~5--10%) to catch judge blind spots
5. **Periodic calibration** -- compare human and judge scores weekly, retune the judge prompt when agreement drops below threshold, focus on specific criteria. some criteria may be more critcial for accuracy either fine tune them or conside combining code instead of llm.